## Abschlussarbeit: Medical Appointment No Shows
Hier bearbeiten wir ein Dataset aus Brasilien: Medical Appointment No Shows:<br>

https://www.kaggle.com/datasets/joniarroba/noshowappointments



Ziel ist es, das Nicht-Wahrnehmen eines Arzttermins vorherzusagen
.<br>


__Es gibt 1 Datei__:<br>

1. KaggleV2-May-2016.csv



### Kriterien für die Bewertung der notebooks (Project-Arbeit)

 - richtiges Anwenden der Besonderheiten des data-sets
 
 - Auswahl richtige loss-Funktion
 
 - Verteilung der target-Values
 
 - feature-engineering
 
 - korrekte Einschätzung der Modell-Performance
 
 - richtiges Anwenden von feature-importance
 
 - grid-search: korrekte Definition des Such-Raums
 
 - Verwendung von Pipelines
 
 - Suche nach bestem Algorithmus (Lern-Algorithmus); mindestens 2 verschieden Ansätze
 

In [3]:
import pandas as pd
noshow = pd.read_csv('../data/KaggleV2-May-2016.csv')

In [5]:
duplicates = noshow[noshow.duplicated(subset="PatientId", keep=False)]

In [7]:
duplicates.shape

(72607, 14)

In [8]:
noshow.shape

(110527, 14)

In [14]:
noshow.loc[noshow['PatientId'] == noshow.PatientId.values[5]]

,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
5,9.598513e+13,5626772,F,2016-04-27T08:36:51Z,2016-04-29T00:00:00Z,76,REPÚBLICA,0,1,0,0,0,0,No
98967,9.598513e+13,5732325,F,2016-05-24T11:19:51Z,2016-06-01T00:00:00Z,76,REPÚBLICA,0,1,0,0,0,0,No


In [15]:
df = noshow.copy()
df["AppointmentDay"] = pd.to_datetime(df["AppointmentDay"])
df = df.sort_values("AppointmentDay")

# Split patients into "train-only" vs "test-only" based on their FIRST appointment
patient_first_date = df.groupby("PatientId")["AppointmentDay"].min().reset_index()
split_date = patient_first_date["AppointmentDay"].quantile(0.8)

train_patients = patient_first_date[patient_first_date["AppointmentDay"] <= split_date]["PatientId"]
test_patients = patient_first_date[patient_first_date["AppointmentDay"] > split_date]["PatientId"]

# Split data based on patient groups (a patient is EITHER in train OR test)
train = df[df["PatientId"].isin(train_patients)]
test = df[df["PatientId"].isin(test_patients)]

# Verify no overlap and correct time split
print(f"Train time range: {train['AppointmentDay'].min()} to {train['AppointmentDay'].max()}")
print(f"Test time range: {test['AppointmentDay'].min()} to {test['AppointmentDay'].max()}")
print(f"Patient overlap: {set(train['PatientId']).intersection(set(test['PatientId']))}") 

Train time range: 2016-04-29 00:00:00+00:00 to 2016-06-08 00:00:00+00:00
Test time range: 2016-05-31 00:00:00+00:00 to 2016-06-08 00:00:00+00:00
Patient overlap: set()


In [32]:
df = df.sort_values("AppointmentDay")

split_date = df["AppointmentDay"].quantile(0.8)
train_patients = df[df["AppointmentDay"] <= split_date]["PatientId"]
test_patients = df[df["AppointmentDay"] > split_date]["PatientId"]

common_patients = set(train_patients).intersection(set(test_patients))
# common_patients = set(df[df["AppointmentDay"] <= split_date]["PatientId"]).intersection(
#                   set(df[df["AppointmentDay"] > split_date]["PatientId"]))

# Remove leaky patients from the test set
test = df[df["PatientId"].isin(test_patients) & ~df["PatientId"].isin(common_patients)]
train = df[df['PatientId'].isin(train_patients) & ~df['PatientId'].isin(common_patients)]

### Watch Out!
There are two difficulties:<br>
 - patients exist several times in the dataset, with different appointment times
 - there's a time ordering - test set should be after the training set
 
__=> eventually, you can not use all 110527 records, because otherwise a clean separation into train and test would not be possible__

